# Interaction Features

**Topic:** Feature Engineering

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import ToggleButtons, Output, VBox

from IPython.display import display, clear_output, HTML

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, KFold

from tkh_utils import PALETTE, FONT, base_layout

---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** what an interaction feature captures and why linear models need them
- **Explain** why the Manhattan-Bronx price gap depends on room type, not just borough
- **Interpret** how the scale of your target — dollars vs. log(dollars) — can make a real interaction effect appear or disappear

> **Tip:** Flip the **target** toggle below. In dollars, the heatmap's rows fan out unevenly across columns — that unevenness is the interaction. Switch to log(price) and the same rows line up proportionally instead, and the interaction's R² gain nearly disappears.

---
## How we got here

In **[03_binning_and_discretization.ipynb](03_binning_and_discretization.ipynb)** you simplified continuous features into groups. But sometimes the problem runs the other direction: two features that look simple individually become rich when you consider them together.

That is what interaction features are for.

---
## Why this matters for data science

Linear models assume each feature contributes independently to the prediction. That assumption breaks constantly in the real world. The dollar value of moving a listing from the Bronx to Manhattan depends heavily on `room_type` — it's a \$172 jump for an entire home, but only a \$49 jump for a shared room.

Without an interaction feature, a linear model has to average those two very different jumps into a single Manhattan coefficient. With one, it can represent both.

---
## Try it yourself

In [2]:
np.random.seed(42)

df = pd.read_csv('../../data/nyc_airbnb.csv')
df = df[df['price'] > 0].copy()
df['log_price'] = np.log1p(df['price'])
df['joint'] = df['neighbourhood_group'] + '_' + df['room_type']

NG_ORDER = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']
ROOM_TYPES = ['Entire home/apt', 'Private room', 'Shared room']

# Dollar-scale summary used in captions regardless of which target is shown
_price_pivot = (df.groupby(['neighbourhood_group', 'room_type'])['price']
                  .mean().unstack('room_type').loc[NG_ORDER, ROOM_TYPES])
PREMIUM_ENTIRE = _price_pivot.loc['Manhattan', 'Entire home/apt'] - _price_pivot.loc['Bronx', 'Entire home/apt']
PREMIUM_SHARED = _price_pivot.loc['Manhattan', 'Shared room'] - _price_pivot.loc['Bronx', 'Shared room']
RATIO_ENTIRE = _price_pivot.loc['Manhattan', 'Entire home/apt'] / _price_pivot.loc['Bronx', 'Entire home/apt']

# R2 for additive-only vs. additive+interaction, computed once per target
_kf = KFold(n_splits=5, shuffle=True, random_state=42)

def _cv_r2(cols, target):
    encode = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cols)])
    pipe = Pipeline([
        ('encode', encode),
        ('scale', StandardScaler(with_mean=False)),
        ('ridge', Ridge()),
    ])
    scores = cross_val_score(pipe, df[cols], df[target], cv=_kf, scoring='r2')
    return scores.mean()

R2_STATS = {}
for _target in ['price', 'log_price']:
    R2_STATS[_target] = {
        'additive': _cv_r2(['neighbourhood_group', 'room_type'], _target),
        'interaction': _cv_r2(['neighbourhood_group', 'room_type', 'joint'], _target),
    }

In [3]:
out_interact = Output()
_initialized = False

target_toggle = ToggleButtons(
    options=[("Price ($)", "price"), ("log(Price)", "log_price")],
    value="price",
    description="Target:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="320px"),
)

def render_interact(change=None):
    global _initialized
    target = target_toggle.value
    label = "price ($)" if target == "price" else "log(price)"

    pivot = df.groupby(['neighbourhood_group', 'room_type'])[target].mean().unstack('room_type')
    pivot = pivot.loc[NG_ORDER, ROOM_TYPES]
    z = pivot.values
    text = [[f"${v:,.0f}" if target == "price" else f"{v:.2f}" for v in row] for row in z]

    fig = go.Figure(
        data=go.Heatmap(
            z=z, x=ROOM_TYPES, y=NG_ORDER, colorscale="Blues",
            text=text, texttemplate="%{text}",
            colorbar=dict(title=label),
        ),
        layout=base_layout(
            title=f"Mean {label} — neighbourhood_group × room_type",
            xaxis_title="room_type", yaxis_title="neighbourhood_group",
        ),
    )

    stats = R2_STATS[target]
    gain = stats['interaction'] - stats['additive']

    if target == "price":
        interp = (
            f"The Manhattan premium is <b>${PREMIUM_ENTIRE:,.0f}</b> for an entire home but only "
            f"<b>${PREMIUM_SHARED:,.0f}</b> for a shared room — one borough number cannot cover "
            f"both, so the interaction feature earns its keep."
        )
    else:
        interp = (
            f"In log space that same premium becomes a roughly constant multiplier "
            f"(Manhattan runs about {RATIO_ENTIRE:.1f}× the Bronx price, for every room type), "
            f"so log(price) already captures it and the interaction adds almost nothing."
        )

    caption = (
        f'<div style="font-family:{FONT["family"]};font-size:13px;background:#f8f9fa;'
        f'padding:12px 16px;border-radius:6px;margin-top:8px;line-height:1.6;color:#333;">'
        f'<p style="margin:0 0 6px 0;"><b>Target = {label}:</b> additive R² = '
        f'{stats["additive"]:.3f}, with the interaction feature = {stats["interaction"]:.3f} '
        f'(gain {gain:+.3f}).</p>'
        f'<p style="margin:0;">{interp}</p>'
        f'</div>'
    )

    with out_interact:
        clear_output(wait=True)
        fig.show()
        display(HTML(caption))
    _initialized = True

target_toggle.observe(render_interact, names="value")
display(VBox([target_toggle, out_interact]))
render_interact()

---
## What's happening?

An interaction feature captures the **joint effect** of two variables — something neither column carries on its own. For categorical features like `neighbourhood_group` and `room_type`, you build it by **concatenating into a joint category, then one-hot encoding**:

```python
airbnb['ng_x_rt'] = airbnb['neighbourhood_group'] + '_' + airbnb['room_type']
```

This creates 15 distinct joint levels — `Manhattan_Entire home/apt`, `Bronx_Shared room`, and so on — each one gets its own one-hot column.

**Never multiply label-encoded categories together.** `LabelEncoder` assigns integers alphabetically, so `room_type_enc * neighbourhood_group_enc` makes every Bronx row (encoded 0) collapse to zero, and treats the numeric gap between `Brooklyn` and `Manhattan` as meaningful when it is just alphabetical order. Two genuinely different combinations can land on the same product by coincidence of their codes. One-hot on the concatenated string avoids this entirely: every real combination gets its own column, and none collide.

Tree-based models (decision trees, random forests, gradient boosting) find interactions automatically through split combinations — a tree can split on `neighbourhood_group` and then split on `room_type` within each branch. Linear models cannot do this on their own; they need the interaction handed to them as an explicit column.

| Feature A | Feature B | Interaction | What it captures | ML models that benefit |
|-----------|-----------|-------------|-------------------|------------------------|
| `neighbourhood_group` | `room_type` | concatenate → one-hot | Location-adjusted room value | Linear, Logistic |
| `latitude` | `longitude` | product | Geographic interaction (weak but real: +0.009 R² on a base of ~0.10) | Linear models only |

### Whether you need this interaction depends on your target's scale

The Manhattan/room-type interaction is real — but it only shows up in the model when the target is raw dollars. Predicting `log1p(price)` instead makes the interaction's R² gain disappear almost entirely. That isn't a bug in the interaction; it's what a log transform *does*: it turns multiplicative effects into additive ones. Manhattan running about 2.4× the Bronx price across every room type is a multiplicative relationship, and `log(2.4 × x) = log(2.4) + log(x)` — a constant additive shift that two separate columns already capture without help.

This connects directly to **[06_handling_skewed_features.ipynb](06_handling_skewed_features.ipynb)**: transforming a skewed target doesn't just fix its distribution, it can change which engineered features actually earn their place in the model.

---
## Real-world example: The Manhattan premium, in dollars

Mean price by neighbourhood group and room type, no transform applied:

| | Entire home/apt | Private room | Shared room |
|---|---|---|---|
| Bronx | \$117.60 | \$60.70 | \$31.80 |
| Manhattan | \$289.20 | \$143.20 | \$80.70 |
| **Manhattan − Bronx** | **+\$171.60** | **+\$82.50** | **+\$48.80** |

The Manhattan premium is not one number. It's \$172 if you're renting an entire home, but only \$49 if you're renting a shared room. A model with only `neighbourhood_group` and `room_type` as separate additive terms must pick a single average Manhattan premium and apply it everywhere — it cannot represent a premium that changes size depending on room type. That's exactly what the interaction feature contributes.

(Notice the *ratio* is nearly constant instead — Manhattan runs about 2.4–2.5× the Bronx price for every room type. That's the multiplicative pattern that log(price) already absorbs, as shown above.)

In [4]:
compare = df[df['neighbourhood_group'].isin(['Bronx', 'Manhattan'])]
bars = (compare.groupby(['neighbourhood_group', 'room_type'])['price']
               .mean().unstack('neighbourhood_group').loc[ROOM_TYPES, ['Bronx', 'Manhattan']])

fig = go.Figure(data=[
    go.Bar(name='Bronx', x=ROOM_TYPES, y=bars['Bronx'], marker_color=PALETTE['muted'],
           text=[f"${v:,.0f}" for v in bars['Bronx']], textposition='outside'),
    go.Bar(name='Manhattan', x=ROOM_TYPES, y=bars['Manhattan'], marker_color=PALETTE['primary'],
           text=[f"${v:,.0f}" for v in bars['Manhattan']], textposition='outside'),
])
fig.update_layout(base_layout(
    title='Mean Price: Bronx vs. Manhattan, by Room Type',
    xaxis_title='room_type', yaxis_title='Mean price ($)',
))
fig.update_layout(barmode='group')
fig.show()

> **Discussion question:** The entire-home gap between Manhattan and the Bronx (\$172) is over three times the shared-room gap (\$49) in absolute dollars. Could a model that only had `neighbourhood_group` and `room_type` as separate features ever learn a \$172 premium for one room type and a \$49 premium for another, using just two additive coefficients? Why not?

---
## Key takeaway

> **An interaction feature expresses what your domain knowledge already knows: the size of one variable's effect can depend on another variable — and whether your model actually needs that interaction spelled out also depends on the scale you chose for your target.**

---
*Next up: 05_polynomial_features — where you add x² and x³ to capture curves that straight lines miss*